In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

print("Booting up Logistics Data Generator v2.0...")

# 1. Configuration
NUM_ORDERS = 50000
hubs = ['Koramangala', 'Indiranagar', 'Whitefield', 'Electronic City']
weather = ['Clear', 'Light Rain', 'Heavy Rain', 'Waterlogged']

# 2. Generating Base Data (Over the last 6 months for 50k volume)
start_date = datetime.now() - timedelta(days=180)
order_timestamps = [start_date + timedelta(minutes=random.randint(0, 259200)) for _ in range(NUM_ORDERS)]
order_timestamps.sort() # Sort chronologically for realism

# Create the base dataframe with uneven probabilities
df = pd.DataFrame({
    'order_id': range(100001, 100001 + NUM_ORDERS),
    'dark_store_name': np.random.choice(hubs, NUM_ORDERS, p=[0.40, 0.30, 0.18, 0.12]), # Koramangala gets most orders
    'customer_lat_lon': [f"12.{random.randint(9000, 9500)}, 77.{random.randint(6000, 6500)}" for _ in range(NUM_ORDERS)],
    'order_timestamp': order_timestamps,
    'cart_value_inr': np.round(np.random.uniform(150.0, 2500.0, NUM_ORDERS), 2),
    'weather_condition': np.random.choice(weather, NUM_ORDERS, p=[0.60, 0.25, 0.10, 0.05])
})

# 3. Simulating Realistic Delivery Times
# Base delivery is 8-12 mins. 
delivery_delays_mins = np.random.uniform(8, 12, NUM_ORDERS)

# Hub Penalties (Whitefield and Electronic City have worse traffic)
delivery_delays_mins = np.where(df['dark_store_name'] == 'Whitefield', delivery_delays_mins + 4, delivery_delays_mins)
delivery_delays_mins = np.where(df['dark_store_name'] == 'Electronic City', delivery_delays_mins + 6, delivery_delays_mins)

# Weather Penalties
delivery_delays_mins = np.where(df['weather_condition'] == 'Light Rain', delivery_delays_mins + 3, delivery_delays_mins)
delivery_delays_mins = np.where(df['weather_condition'] == 'Heavy Rain', delivery_delays_mins + 8, delivery_delays_mins)
delivery_delays_mins = np.where(df['weather_condition'] == 'Waterlogged', delivery_delays_mins + 15, delivery_delays_mins)

# 4. Creating the completion timestamp
df['delivery_completion_timestamp'] = df['order_timestamp'] + pd.to_timedelta(delivery_delays_mins, unit='m')

# 5. Injecting system errors (2% missing timestamps for SQL cleaning practice)
missing_indices = df.sample(frac=0.02).index
df.loc[missing_indices, 'delivery_completion_timestamp'] = pd.NaT

# 6. Save to CSV
df.to_csv('raw_delivery_logs_50k.csv', index=False)
print(f"Success! Generated raw_delivery_logs_50k.csv with {NUM_ORDERS} records.")

Booting up Logistics Data Generator v2.0...
Success! Generated raw_delivery_logs_50k.csv with 50000 records.
